# Baseline salinity + crop productivity (in-notebook Solara)

Reuses `solara_mekong` map tools from IDP-workbench:

| Page | Data |
|------|------|
| **Hazard** | Remote GeoServer WMS `baseline_p50_{2014\|2015\|2016}` (workspace `salinity`) |
| **Impact** | Crop productivity correction GeoParquet (13_crop_productivity_correction) |

**Kernel:** **Python (solara-mekong)**

**Prerequisites**
1. Hazard WMS uses **remote** Deltares GeoServer (VPN usually required).
2. Crop parquet on C: under `...\IDP\Data\stac_folder\crop_productivity_correction\` or public GCS fallback.
3. Run cells top to bottom.


## 1) Setup (path, light theme)


In [1]:
import sys
import warnings
from pathlib import Path

import solara

warnings.filterwarnings("ignore", category=FutureWarning, module="google")

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

solara.lab.theme.dark = False

print("REPO_ROOT:", REPO_ROOT)
print("Solara dark theme:", solara.lab.theme.dark)


REPO_ROOT: C:\Users\fuentesm\Deltares\2024B\IDP\Tools\Repositories\IDP-workbench\dashboards\Salinity_Intrusion_Mekong_Dashboard\solara_mekong
Solara dark theme: False


## 2) Connectivity checks

In [2]:
import requests
from solara_mekong.utils.general import (
    BASELINE_YEAR_OPTIONS,
    REMOTE_GEOSERVER_URL,
    get_baseline_salinity_wms_config,
    resolve_crop_parquet_path,
)

try:
    r = requests.get(f"{REMOTE_GEOSERVER_URL}/web/", timeout=15, verify=False)
    print(f"Remote GeoServer UI: {r.status_code} ({REMOTE_GEOSERVER_URL}/web/)")
except Exception as exc:
    print("Remote GeoServer not reachable (VPN may be required):", exc)

for y in BASELINE_YEAR_OPTIONS:
    cfg = get_baseline_salinity_wms_config(y)
    print(f"  {y}: {cfg['layer']} @ {cfg['url']}")

print("Crop parquet:", resolve_crop_parquet_path())


c:\Users\fuentesm\AppData\Local\miniforge3\envs\solara_mekong\lib\site-packages\pystac_client\client.py:191: NoConformsTo: Server does not advertise any conformance classes.
  warnings.warn(NoConformsTo())
c:\Users\fuentesm\AppData\Local\miniforge3\envs\solara_mekong\lib\site-packages\pystac_client\client.py:410: FallbackToPystac: Falling back to pystac. This might be slow.
  self._warn_about_fallback("COLLECTIONS", "FEATURES")
c:\Users\fuentesm\AppData\Local\miniforge3\envs\solara_mekong\lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'international-delta-platform.avi.directory.intra'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\fuentesm\AppData\Local\miniforge3\envs\solara_mekong\lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '

Remote GeoServer UI: 200 (https://international-delta-platform.avi.directory.intra/geoserver/web/)
  2014: baseline_p50_2014 @ https://international-delta-platform.avi.directory.intra/geoserver/wms/salinity
  2015: baseline_p50_2015 @ https://international-delta-platform.avi.directory.intra/geoserver/wms/salinity
  2016: baseline_p50_2016 @ https://international-delta-platform.avi.directory.intra/geoserver/wms/salinity
Crop parquet: C:\Ocean\Work\Projects\2026\IDP\Data\stac_folder\crop_productivity_correction\parquets\baseline\corrected_yield.parquet


## 3) Hazard — salinity (2014 / 2015 / 2016)

Uses `salinity_hazard.Page` → `general.get_baseline_salinity_wms_config` + `map.Map` (same stack as `hazard.py` / `view_dashboard`).


In [3]:
from solara_mekong.pages.salinity_hazard import Page as SalinityHazardPage

SalinityHazardPage()


Cannot show ipywidgets in text

## 4) Impact — crop productivity correction

Freshwater-zone choropleth from `13_crop_productivity_correction` GeoParquet  
(same product as `Food-Security/scripts/compare_crop_productivity_local_vs_stac.ipynb`).

Controls: **year**, **rice season**, **metric** (`corrected_yield`, `hectares`, `salinity`, …).

In [4]:
from solara_mekong.pages.crop_impact import Page as CropImpactPage

CropImpactPage()

Cannot show ipywidgets in text

## Notes

- Hazard uses remote layers `baseline_p50_2014/2015/2016` (Deltares VPN may be required).
- Impact polygons are **freshwater zones** (`province_fwzone`), not the Solara province (`provc.geojson`) layer used on the 2050 impact page.
- After editing `solara_mekong` Python modules, restart the kernel before re-running.
